# 2. 临床基线、单因素与多因素分析

读取步骤 1 的同一份患者划分。基线表保留缺失计数，不先填补原始数据。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import ov_radiomics as rad

# Run notebooks in order, with examples/ as the working directory.
output = Path("output/notebook_demo")

In [ ]:
clinical = pd.read_csv(output / "clinical.csv")
features = ["age", "sex", "marker"]
continuous = ["age", "marker"]
categorical = ["sex"]
clinical.head()

`split="group"` 表示在 train/test 各自内部比较结局类别，不是检验 train 与 test 的差异。连续变量可选 Mann–Whitney 或 Welch，分类变量按设置使用 Fisher/卡方。

In [ ]:
baseline = rad.baseline_table(
    clinical, target="label", features=features,
    continuous=continuous, categorical=categorical, split="group",
    continuous_test="mannwhitney", categorical_test="auto",
)
baseline.to_csv(output / "baseline.csv", index=False)
baseline

## 训练集的 Logistic 推断
显式设置分类变量参考组与 95% CI。采用完整病例分析，返回使用/删除的样本数；完全分离、秩不足或不收敛会报错。

这里的非惩罚 Logit 用于 OR/CI 推断，与预测用的 sklearn LR 区分。示例预先指定临床变量；不根据测试组 p 值选择变量。

In [ ]:
train = clinical.loc[clinical.group == "train"]
univariate = rad.univariate_logistic(
    train, "label", features, categorical=categorical,
    reference={"sex": "F"}, confidence=0.95,
)
univariate.to_csv(output / "univariate.csv", index=False)
univariate

In [ ]:
multivariable = rad.multivariable_logistic(
    train, "label", features, categorical=categorical,
    reference={"sex": "F"}, confidence=0.95,
)
print("Used:", multivariable.n_used, "Dropped:", multivariable.n_dropped)
multivariable.table.to_csv(output / "multivariable.csv", index=False)
multivariable.table

若实际研究根据单因素结果筛选临床变量，应只使用训练数据；若还要报告筛选后模型的交叉验证性能，变量筛选本身也必须纳入交叉验证。这里不提供自动逐步回归。